# Machine Learning - Lab 03
## Decision Trees: Prediction and Training using Mutual Information

Activities covered: 1) Entropy, 2) Mutual Information, 3) Prediction, 4) Recursive Training, 5) Testing the trained tree.
Everything is plain Python (no scikit-learn).

In [1]:
import math
from collections import Counter

## Activity 1: Implement Entropy
Entropy measures uncertainty (impurity) of a set of labels: $H = -\sum p \log_2 p$.
A pure set has entropy 0; a 50/50 set has entropy 1.

In [2]:
def entropy(labels):
    if len(labels) == 0:
        return 0
    total = len(labels)
    result = 0
    for count in Counter(labels).values():
        p = count / total
        result -= p * math.log2(p)
    return result

labels1 = ["Pass", "Pass", "Pass", "Pass"]
labels2 = ["Pass", "Pass", "Fail", "Fail"]
labels3 = ["Pass", "Fail", "Pass", "Fail"]

print("labels1:", entropy(labels1))
print("labels2:", entropy(labels2))
print("labels3:", entropy(labels3))

labels1: 0.0
labels2: 1.0
labels3: 1.0


**Why the values differ**
- `labels1` has only "Pass", so there is no uncertainty: entropy = **0**.
- `labels2` is half Pass, half Fail: maximum uncertainty for two classes, entropy = **1**.
- `labels3` has the same 50/50 mix as `labels2` (only the order differs). Entropy depends only on the **proportions**, not the order, so it is also **1**.

## The dataset

In [3]:
dataset = [
    {"Attendance": "High", "Study": "High", "Assignment": "Yes", "Label": "Pass"},
    {"Attendance": "High", "Study": "Low",  "Assignment": "Yes", "Label": "Pass"},
    {"Attendance": "High", "Study": "Low",  "Assignment": "No",  "Label": "Pass"},
    {"Attendance": "Low",  "Study": "High", "Assignment": "Yes", "Label": "Pass"},
    {"Attendance": "Low",  "Study": "Low",  "Assignment": "No",  "Label": "Fail"},
    {"Attendance": "Low",  "Study": "Low",  "Assignment": "Yes", "Label": "Fail"},
    {"Attendance": "High", "Study": "High", "Assignment": "No",  "Label": "Pass"},
    {"Attendance": "Low",  "Study": "High", "Assignment": "No",  "Label": "Fail"},
]
features = ["Attendance", "Study", "Assignment"]

## Activity 2: Conditional Entropy and Mutual Information
- Conditional entropy `H(Y|X)` = weighted average of the entropy inside each group of feature `X`.
- Mutual information `I(Y;X) = H(Y) - H(Y|X)`. A higher value means a more useful feature.

In [5]:
def conditional_entropy(dataset, feature, label_col="Label"):
    total = len(dataset)
    result = 0
    for value in sorted(set(row[feature] for row in dataset)):
        group = [row[label_col] for row in dataset if row[feature] == value]
        weight = len(group) / total
        result += weight * entropy(group)
    return result

def mutual_information(dataset, feature, label_col="Label"):
    labels = [row[label_col] for row in dataset]
    return entropy(labels) - conditional_entropy(dataset, feature, label_col)

for f in features:
    print(f"MI({f}) =", round(mutual_information(dataset, f), 3))

best_feature = max(features, key=lambda f: mutual_information(dataset, f))
print("Best feature (root):", best_feature)

MI(Attendance) = 0.549
MI(Study) = 0.049
MI(Assignment) = 0.049
Best feature (root): Attendance


### Hand trace of one training step (Attendance)

In [6]:
labels = [row["Label"] for row in dataset]
print("Label counts:", dict(Counter(labels)))
print("H(Y) =", round(entropy(labels), 3))

for value in ["High", "Low"]:
    group = [row["Label"] for row in dataset if row["Attendance"] == value]
    print("Attendance =", value, "->", group, "-> entropy =", round(entropy(group), 3))

print("H(Y|Attendance) =", round(conditional_entropy(dataset, "Attendance"), 3))
print("I(Y;Attendance) = H(Y) - H(Y|Attendance) =", round(mutual_information(dataset, "Attendance"), 3))

Label counts: {'Pass': 5, 'Fail': 3}
H(Y) = 0.954
Attendance = High -> ['Pass', 'Pass', 'Pass', 'Pass'] -> entropy = 0.0
Attendance = Low -> ['Pass', 'Fail', 'Fail', 'Fail'] -> entropy = 0.811
H(Y|Attendance) = 0.406
I(Y;Attendance) = H(Y) - H(Y|Attendance) = 0.549


Attendance has the highest mutual information, so it becomes the **root**. Students with High attendance are all "Pass" (a pure group), while Low attendance is still mixed and needs more splits.

## Activity 3: Implement Prediction
A leaf stores a label. An internal node stores a feature and one child per feature value. To predict, start at the root and follow the branch matching the example until a leaf is reached.

In [7]:
class Node:
    def __init__(self, node_type, label=None, feature=None):
        self.type = node_type      # "leaf" or "internal"
        self.label = label         # used by leaf nodes
        self.feature = feature     # used by internal nodes
        self.branches = {}         # feature value -> child Node

def predict(tree, x):
    current = tree
    while current.type != "leaf":
        current = current.branches[x[current.feature]]
    return current.label

Build the small tree from the manual by hand (Attendance, then Study) and test three new students.

In [8]:
fail_leaf = Node("leaf", label="Fail")
pass_leaf = Node("leaf", label="Pass")

study_node = Node("internal", feature="Study")
study_node.branches["Low"] = fail_leaf
study_node.branches["High"] = pass_leaf

root = Node("internal", feature="Attendance")
root.branches["Low"] = fail_leaf
root.branches["High"] = study_node

students = [
    {"Attendance": "High", "Study": "High"},
    {"Attendance": "High", "Study": "Low"},
    {"Attendance": "Low",  "Study": "High"},
]
for s in students:
    print(s, "->", predict(root, s))

{'Attendance': 'High', 'Study': 'High'} -> Pass
{'Attendance': 'High', 'Study': 'Low'} -> Fail
{'Attendance': 'Low', 'Study': 'High'} -> Fail


## Activity 4: Implement Recursive Training
`train()` follows the lecture algorithm:
1. **Base case:** all labels identical -> leaf with that label.
2. **Base case:** no features left -> leaf with the majority label.
3. Otherwise pick the feature with the **highest mutual information**, split the data by its values, train a child tree on each subset (recursion).

In [9]:
def majority_label(dataset, label_col="Label"):
    labels = [row[label_col] for row in dataset]
    return Counter(labels).most_common(1)[0][0]

def train(dataset, features, label_col="Label"):
    labels = [row[label_col] for row in dataset]

    if len(set(labels)) == 1:                     # base case 1
        return Node("leaf", label=labels[0])
    if len(features) == 0:                        # base case 2
        return Node("leaf", label=majority_label(dataset, label_col))

    best = max(features, key=lambda f: mutual_information(dataset, f, label_col))
    node = Node("internal", feature=best)
    remaining = [f for f in features if f != best]

    for value in sorted(set(row[best] for row in dataset)):
        subset = [row for row in dataset if row[best] == value]
        node.branches[value] = train(subset, remaining, label_col)
    return node

tree = train(dataset, features)

Print the trained tree to see the learned rules:

In [10]:
def print_tree(node, indent=""):
    if node.type == "leaf":
        print(indent + "->", node.label)
        return
    for value, child in node.branches.items():
        print(indent + node.feature + " = " + value)
        print_tree(child, indent + "    ")

print_tree(tree)

Attendance = High
    -> Pass
Attendance = Low
    Study = High
        Assignment = No
            -> Fail
        Assignment = Yes
            -> Pass
    Study = Low
        -> Fail


In the *Low attendance* branch, Study and Assignment have equal mutual information (a tie). `max()` returns the first one in the list, so **Study** is chosen.

## Activity 5: Test the Trained Tree

In [ ]:
new_students = [
    {"Attendance": "High", "Study": "Low",  "Assignment": "No"},
    {"Attendance": "Low",  "Study": "Low",  "Assignment": "Yes"},
    {"Attendance": "Low",  "Study": "High", "Assignment": "Yes"},
    {"Attendance": "Low",  "Study": "High", "Assignment": "No"},
    {"Attendance": "High", "Study": "High", "Assignment": "Yes"},
]
for s in new_students:
    print(s, "->", predict(tree, s))

Check: the tree should reproduce the labels of the training data it was built from.

In [11]:
correct = sum(predict(tree, row) == row["Label"] for row in dataset)
print("Training accuracy:", correct, "/", len(dataset))

Training accuracy: 8 / 8
